In [1]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import os
import torch
import numpy as np
from torchvision import transforms as T
from torchvision.transforms import Normalize
from transformers import (
    ViTImageProcessor, 
    ViTForImageClassification,
    TrainingArguments, 
    Trainer,
)
import evaluate

/Users/maximdorogov/anaconda3/envs/tryolabs/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Definimos los datasets que vamos a utilizar para modelar los datos de entrenamiento y evaluacion.

In [ ]:
class PlantGrassDataset(torch.utils.data.Dataset):

    IMG_CLASS_COL = 1

    def __init__(self, dataset_csv:str, folder_path: str, transform=None):
        """
        Parameters
        ----------
        dataset_csv:str
            Path to a csv file contaning image name and class labels
        folder_path:str
            Path to a folder with all the images
        """
        super().__init__()

        self.transform = transform
        self._folder_path = folder_path
        self._dataset_df = pd.read_csv(dataset_csv, header=None)
        self.labels = list(set(self._dataset_df[self.IMG_CLASS_COL]))
        self.label2id = {label:i for i, label in enumerate(self.labels)}
        self.id2label = {i:label for i, label in enumerate(self.labels)}

    def __getitem__(self, index):
        # store as attributes for further usage
        self.image_name, self.class_name = self._dataset_df.loc[index]
        image = Image.open(os.path.join(self._folder_path, self.image_name))
        image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "label": self.label2id[self.class_name]}

    def __len__(self):
        return len(self._dataset_df)

In [3]:
# Path al archivo csv que lista las imágenes y sus etiquetas
INPUT_CSV_FILE = "./data/all_images.csv"
# Path al directorio que contiene las imágenes
IMAGE_FOLDER = "./data/images"
# Path al directorio donde se guardarán los resultados del entrenamiento
EXPERIMENT_FOLDER = "./training_results"
# Nombre del transformer a entrenar
MODEL_NAME = "WinKawaks/vit-tiny-patch16-224"

Instanciamos el dataset y definimos las transformaciones para data augmentation

In [4]:
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

transforms = T.Compose([
    T.ToTensor(),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.Resize((processor.size["height"], processor.size["width"])),
    Normalize(mean=processor.image_mean, std=processor.image_std)
])

main_dataset = PlantGrassDataset(
    dataset_csv=INPUT_CSV_FILE,
    folder_path=IMAGE_FOLDER,
    transform=transforms,
)

Separamos los datos en test y train, creamos las carpetas para guardar los resultados del entrenamiento y la metadata

In [5]:
# split main dataset
val_perc = 0.2
train_data, val_data = torch.utils.data.random_split(
    main_dataset, [1 - val_perc, val_perc])

# export image filenames and classes used for validation
data_for_export = [(val_data.dataset.image_name, val_data.dataset.class_name) 
                    for _, _ in val_data]

os.makedirs(EXPERIMENT_FOLDER, exist_ok=True)
output_path = os.path.join(EXPERIMENT_FOLDER, 'val_data.csv')
data_for_export = pd.DataFrame(data_for_export)
data_for_export.to_csv(output_path, index=False, header=False)

print(f'Train samples: {len(train_data)}\nTest samples: {len(val_data)}')

Train samples: 3800
Test samples: 950


Entrenamiento

In [6]:
EPOCHS = 10
BATCH_SIZE = 4

In [7]:
# Definimos la metrica a utilizar para la evaluacion

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [10]:
# solo para macbooks con apple silicon
print(torch.backends.mps.is_available())
device = "mps" if torch.backends.mps.is_available() else "cpu"
use_mps = True if device == "mps" else False

True


In [ ]:
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    id2label=main_dataset.id2label,
    label2id=main_dataset.label2id,
    ignore_mismatched_sizes=True).to('cpu')
training_args = TrainingArguments(
    output_dir=EXPERIMENT_FOLDER,
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    use_mps_device=use_mps,  # Set to True if using MPS on Mac
    learning_rate=5e-4,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=processor,
    compute_metrics=compute_metrics,
)
trainer.train()